# Module 4 • Distributional Semantics and Word Embeddings

# Lesson 21 • Distributional Semantics and Word Embedding Foundations

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 100–130 minutes

---

## Scope

This lesson introduces the distributional hypothesis and explains how words
can be represented as vectors derived from their contexts. It develops
count-based distributional representations, co-occurrence matrices,
similarity measures, dimensionality reduction, dense embeddings, nearest
neighbors, analogy intuition, evaluation, and multilingual considerations.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain the distributional hypothesis;
- distinguish one-hot, sparse count, and dense vector representations;
- build a word-context co-occurrence matrix;
- explain context windows and weighting choices;
- calculate cosine similarity between word vectors;
- identify nearest neighbors in a vector space;
- explain Pointwise Mutual Information and Positive PMI;
- reduce sparse vectors with Truncated SVD;
- interpret dense embedding dimensions cautiously;
- explain the intuition behind word analogies;
- distinguish intrinsic and extrinsic embedding evaluation;
- identify frequency, domain, and bias effects;
- discuss Arabic and multilingual embedding challenges.

## Table of Contents

1. From Sparse Features to Vector Meaning
2. The Distributional Hypothesis
3. One-Hot Representations
4. Distributional Contexts
5. Context Windows
6. Building a Co-Occurrence Matrix
7. Interpreting Co-Occurrence Vectors
8. Vector Similarity
9. Cosine Similarity
10. Nearest Neighbors
11. Frequency Bias
12. PMI and PPMI Weighting
13. Dimensionality Reduction
14. Dense Embeddings
15. Visualizing Embedding Spaces
16. Word Analogies
17. Polysemy and Context Dependence
18. Intrinsic Evaluation
19. Extrinsic Evaluation
20. Bias and Ethical Considerations
21. Domain and Corpus Effects
22. Arabic and Multilingual Considerations
23. Reproducibility and Reporting
24. Knowledge Check
25. Exercises
26. Summary and Next Lesson

# 1. From Sparse Features to Vector Meaning

Earlier lessons represented documents using sparse features such as:

- Bag-of-Words;
- TF-IDF;
- word n-grams;
- character n-grams.

These representations are effective, but they usually treat each vocabulary
item as an independent feature.

Distributional semantics asks a different question:

> Can the meaning of a word be approximated from the contexts in which it
> appears?

In [ ]:
import pandas as pd

representation_comparison = pd.DataFrame(
    [
        ("One-hot", "one position per word", "very sparse", "no learned similarity"),
        ("Count-based", "context frequencies", "sparse", "similar contexts become similar"),
        ("Dense embedding", "learned compact vector", "dense", "graded geometric similarity"),
    ],
    columns=["Representation", "Main idea", "Structure", "Semantic behavior"],
)

representation_comparison

# 2. The Distributional Hypothesis

The **distributional hypothesis** proposes that words appearing in similar
contexts tend to have related meanings.

Example:

```text
The doctor treated the patient.
The nurse examined the patient.
```

`doctor` and `nurse` occur in related contexts and may receive similar vectors.

Distributional similarity does not guarantee synonymy. It may capture:

- semantic similarity;
- topical relatedness;
- syntactic similarity;
- domain association;
- social or corpus-specific patterns.

In [ ]:
relation_examples = pd.DataFrame(
    [
        ("doctor", "physician", "semantic similarity"),
        ("doctor", "hospital", "topical relatedness"),
        ("run", "walk", "syntactic and semantic similarity"),
        ("bank", "loan", "domain association"),
    ],
    columns=["Word 1", "Word 2", "Possible relation"],
)

relation_examples

# 3. One-Hot Representations

A one-hot vector contains one `1` and zeros elsewhere.

Vocabulary:

```text
doctor, hospital, nurse, patient
```

One-hot vectors:

```text
doctor   → [1, 0, 0, 0]
hospital → [0, 1, 0, 0]
nurse    → [0, 0, 1, 0]
patient  → [0, 0, 0, 1]
```

In [ ]:
import numpy as np

vocabulary = ["doctor", "hospital", "nurse", "patient"]

one_hot = np.eye(len(vocabulary), dtype=int)

one_hot_frame = pd.DataFrame(
    one_hot,
    index=vocabulary,
    columns=vocabulary,
)

one_hot_frame

Any two different one-hot vectors are orthogonal, so one-hot representation
provides no graded similarity between distinct words.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

one_hot_similarity = cosine_similarity(one_hot)

pd.DataFrame(
    one_hot_similarity,
    index=vocabulary,
    columns=vocabulary,
)

# 4. Distributional Contexts

A word can be represented by the words that occur near it.

Example corpus:

```text
the doctor treated the patient
the nurse examined the patient
the doctor worked at the hospital
the nurse worked at the clinic
```

In [ ]:
corpus = [
    "the doctor treated the patient",
    "the nurse examined the patient",
    "the doctor worked at the hospital",
    "the nurse worked at the clinic",
    "the surgeon treated the patient",
    "the physician worked at the hospital",
    "the teacher taught the student",
    "the professor taught the student",
    "the teacher worked at the school",
    "the professor worked at the university",
]

tokenized_corpus = [
    sentence.lower().split()
    for sentence in corpus
]

tokenized_corpus[:3]

Possible context definitions include:

- words inside a fixed window;
- words in the same sentence;
- dependency-linked words;
- document-level co-occurrence;
- syntactic roles;
- subword units.

# 5. Context Windows

A **context window** determines how many neighboring tokens contribute to a
target word representation.

For window size 2:

```text
the doctor treated the patient
    ↑      target
```

Context for `doctor` may include:

```text
the, treated, the
```
depending on boundary and duplicate-counting policy.

In [ ]:
def extract_contexts(
    tokens: list[str],
    window_size: int,
) -> list[tuple[str, str]]:
    if window_size <= 0:
        raise ValueError("window_size must be positive")

    pairs = []

    for target_index, target in enumerate(tokens):
        left = max(0, target_index - window_size)
        right = min(len(tokens), target_index + window_size + 1)

        for context_index in range(left, right):
            if context_index == target_index:
                continue

            pairs.append((target, tokens[context_index]))

    return pairs


extract_contexts(
    "the doctor treated the patient".split(),
    window_size=2,
)

Smaller windows often capture syntactic similarity. Larger windows often
capture broader topical relatedness.

# 6. Building a Co-Occurrence Matrix

A co-occurrence matrix contains:

- one row per target word;
- one column per context word;
- one count per target-context pair.

In [ ]:
from collections import Counter

corpus_vocabulary = sorted({
    token
    for sentence in tokenized_corpus
    for token in sentence
})

word_to_index = {
    word: index
    for index, word in enumerate(corpus_vocabulary)
}

cooccurrence = np.zeros(
    (len(corpus_vocabulary), len(corpus_vocabulary)),
    dtype=float,
)

window_size = 2

for sentence in tokenized_corpus:
    for target, context in extract_contexts(sentence, window_size):
        cooccurrence[
            word_to_index[target],
            word_to_index[context],
        ] += 1

cooccurrence_frame = pd.DataFrame(
    cooccurrence,
    index=corpus_vocabulary,
    columns=corpus_vocabulary,
)

cooccurrence_frame.loc[
    ["doctor", "nurse", "teacher", "professor"],
    ["patient", "worked", "hospital", "student", "school", "university"],
]

Rows of the matrix are distributional word vectors.

# 7. Interpreting Co-Occurrence Vectors

Words with similar contexts should have similar rows.

In [ ]:
selected_words = [
    "doctor",
    "nurse",
    "surgeon",
    "physician",
    "teacher",
    "professor",
]

selected_contexts = [
    "treated",
    "patient",
    "worked",
    "hospital",
    "taught",
    "student",
    "school",
    "university",
]

cooccurrence_frame.loc[
    selected_words,
    selected_contexts,
]

The small corpus produces noisy and incomplete vectors. Large corpora provide
richer context statistics.

# 8. Vector Similarity

Similarity functions compare two vectors.

Common choices include:

- cosine similarity;
- dot product;
- Euclidean distance;
- Manhattan distance;
- correlation.

For distributional vectors, cosine similarity is common because it focuses on
direction rather than raw magnitude.

# 9. Cosine Similarity

For vectors \(a\) and \(b\):

\[
cosine(a,b)
=
\frac{a \cdot b}
{||a||\,||b||}
\]

Typical interpretation:

- close to 1: similar direction;
- close to 0: little overlap;
- below 0: opposite direction in spaces that contain negative values.

In [ ]:
def cosine_between_words(
    word_a: str,
    word_b: str,
    matrix: np.ndarray = cooccurrence,
) -> float:
    vector_a = matrix[word_to_index[word_a]].reshape(1, -1)
    vector_b = matrix[word_to_index[word_b]].reshape(1, -1)

    if np.linalg.norm(vector_a) == 0 or np.linalg.norm(vector_b) == 0:
        return 0.0

    return float(
        cosine_similarity(vector_a, vector_b)[0, 0]
    )


comparison_pairs = [
    ("doctor", "nurse"),
    ("doctor", "teacher"),
    ("teacher", "professor"),
    ("hospital", "school"),
]

for left, right in comparison_pairs:
    print(
        f"{left:<10} {right:<10} "
        f"{cosine_between_words(left, right):.3f}"
    )

Similarity values are corpus-dependent. A score has meaning only relative to
the representation and comparison set.

# 10. Nearest Neighbors

Nearest neighbors are the words with the highest similarity to a target word.

In [ ]:
def nearest_neighbors(
    target_word: str,
    matrix: np.ndarray,
    words: list[str],
    top_k: int = 5,
) -> pd.DataFrame:
    if target_word not in word_to_index:
        raise KeyError(f"Unknown word: {target_word}")

    target_vector = matrix[
        word_to_index[target_word]
    ].reshape(1, -1)

    scores = cosine_similarity(
        target_vector,
        matrix,
    ).ravel()

    rows = []

    for index, score in enumerate(scores):
        word = words[index]

        if word == target_word:
            continue

        rows.append(
            {
                "word": word,
                "similarity": float(score),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["similarity", "word"],
            ascending=[False, True],
        )
        .head(top_k)
        .reset_index(drop=True)
    )


nearest_neighbors(
    "doctor",
    cooccurrence,
    corpus_vocabulary,
    top_k=6,
)

Nearest neighbors should be interpreted manually. They may be synonyms,
co-hyponyms, topical associates, or artifacts.

# 11. Frequency Bias

Frequent words tend to have larger co-occurrence counts and may dominate
similarity.

In [ ]:
word_frequency = Counter(
    token
    for sentence in tokenized_corpus
    for token in sentence
)

frequency_frame = pd.DataFrame(
    word_frequency.items(),
    columns=["word", "frequency"],
).sort_values(
    "frequency",
    ascending=False,
)

frequency_frame.head(12)

Weighting schemes can reduce the influence of universally frequent contexts.

# 12. PMI and PPMI Weighting

**Pointwise Mutual Information (PMI)** compares observed co-occurrence with
expected co-occurrence under independence.

\[
PMI(w,c)
=
\log_2
\frac{P(w,c)}
{P(w)P(c)}
\]

**Positive PMI (PPMI)** replaces negative PMI values with zero.

In [ ]:
def ppmi_matrix(
    counts: np.ndarray,
    smoothing: float = 1e-12,
) -> np.ndarray:
    total = counts.sum()

    row_sums = counts.sum(axis=1, keepdims=True)
    column_sums = counts.sum(axis=0, keepdims=True)

    expected = (
        row_sums @ column_sums
        / max(total, smoothing)
    )

    ratio = (
        counts * total
        / np.maximum(
            row_sums @ column_sums,
            smoothing,
        )
    )

    with np.errstate(divide="ignore"):
        pmi = np.log2(
            np.maximum(ratio, smoothing)
        )

    pmi[counts == 0] = 0.0

    return np.maximum(pmi, 0.0)


ppmi = ppmi_matrix(cooccurrence)

ppmi_frame = pd.DataFrame(
    ppmi,
    index=corpus_vocabulary,
    columns=corpus_vocabulary,
)

ppmi_frame.loc[
    ["doctor", "nurse", "teacher", "professor"],
    selected_contexts,
].round(3)

PPMI emphasizes informative associations but can overemphasize rare events in
small corpora.

In [ ]:
for left, right in comparison_pairs:
    score = cosine_between_words(
        left,
        right,
        matrix=ppmi,
    )

    print(
        f"{left:<10} {right:<10} "
        f"PPMI cosine={score:.3f}"
    )

# 13. Dimensionality Reduction

Co-occurrence vectors have one dimension per context feature and are often
large and sparse.

**Truncated Singular Value Decomposition (SVD)** projects them into a smaller
dense space.

In [ ]:
from sklearn.decomposition import TruncatedSVD

embedding_dimension = 4

svd = TruncatedSVD(
    n_components=embedding_dimension,
    random_state=42,
)

dense_embeddings = svd.fit_transform(ppmi)

print("Original shape:", ppmi.shape)
print("Dense shape:", dense_embeddings.shape)
print(
    "Explained variance:",
    round(float(svd.explained_variance_ratio_.sum()), 3),
)

In [ ]:
dense_embedding_frame = pd.DataFrame(
    dense_embeddings,
    index=corpus_vocabulary,
    columns=[
        f"dimension_{index}"
        for index in range(embedding_dimension)
    ],
)

dense_embedding_frame.loc[selected_words].round(3)

Individual SVD dimensions usually do not have simple human-readable meanings.
Semantic patterns emerge from combinations of dimensions.

# 14. Dense Embeddings

A **dense embedding** maps each vocabulary item to a compact real-valued
vector.

Advantages include:

- lower dimensionality;
- graded similarity;
- reusable features;
- better generalization than one-hot vectors.

Limitations include:

- corpus dependence;
- one vector per word type in static embeddings;
- inherited social bias;
- weak handling of unseen words without subword methods.

In [ ]:
nearest_neighbors(
    "doctor",
    dense_embeddings,
    corpus_vocabulary,
    top_k=6,
)

Count-based SVD embeddings and predictive neural embeddings are different
methods, but both use distributional context.

# 15. Visualizing Embedding Spaces

Two-dimensional projections can support exploration, but they distort the
original geometry.

In [ ]:
visualization_svd = TruncatedSVD(
    n_components=2,
    random_state=42,
)

embedding_2d = visualization_svd.fit_transform(ppmi)

visualization_frame = pd.DataFrame(
    {
        "word": corpus_vocabulary,
        "x": embedding_2d[:, 0],
        "y": embedding_2d[:, 1],
    }
)

visualization_frame.head()

In [ ]:
import matplotlib.pyplot as plt

words_to_plot = [
    "doctor", "nurse", "surgeon", "physician",
    "teacher", "professor", "student",
    "hospital", "clinic", "school", "university",
]

plot_data = visualization_frame[
    visualization_frame["word"].isin(words_to_plot)
]

plt.figure(figsize=(9, 6))
plt.scatter(
    plot_data["x"],
    plot_data["y"],
)

for row in plot_data.itertuples(index=False):
    plt.text(
        row.x,
        row.y,
        row.word,
    )

plt.title("Two-Dimensional Projection of Distributional Vectors")
plt.xlabel("Projection dimension 1")
plt.ylabel("Projection dimension 2")
plt.tight_layout()
plt.show()

A visually close pair in two dimensions may not be equally close in the
original embedding space.

# 16. Word Analogies

Static embedding spaces may exhibit approximate vector relations:

```text
king - man + woman ≈ queen
```

This is an empirical pattern, not a guaranteed algebraic law.

In [ ]:
def analogy(
    positive_a: str,
    negative: str,
    positive_b: str,
    matrix: np.ndarray,
    words: list[str],
    top_k: int = 5,
) -> pd.DataFrame:
    required = [positive_a, negative, positive_b]

    for word in required:
        if word not in word_to_index:
            raise KeyError(f"Unknown word: {word}")

    target = (
        matrix[word_to_index[positive_a]]
        - matrix[word_to_index[negative]]
        + matrix[word_to_index[positive_b]]
    ).reshape(1, -1)

    scores = cosine_similarity(
        target,
        matrix,
    ).ravel()

    excluded = set(required)
    rows = []

    for index, score in enumerate(scores):
        word = words[index]

        if word in excluded:
            continue

        rows.append(
            {
                "word": word,
                "similarity": float(score),
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["similarity", "word"],
            ascending=[False, True],
        )
        .head(top_k)
        .reset_index(drop=True)
    )

The lesson corpus is too small for meaningful analogies. The function is
included to demonstrate the computation.

In [ ]:
analogy(
    "doctor",
    "hospital",
    "school",
    dense_embeddings,
    corpus_vocabulary,
    top_k=5,
)

Analogy performance is sensitive to vocabulary, corpus, preprocessing, and
evaluation design.

# 17. Polysemy and Context Dependence

Static word embeddings assign one vector to each word type.

Example:

```text
bank of the river
bank approved the loan
```

A single static vector for `bank` mixes financial and geographic senses.

In [ ]:
polysemy_examples = pd.DataFrame(
    [
        ("bank", "financial institution", "The bank approved the loan."),
        ("bank", "river edge", "They walked along the river bank."),
        ("bat", "animal", "A bat flew from the cave."),
        ("bat", "sports equipment", "The player held the bat."),
    ],
    columns=["Word", "Sense", "Context"],
)

polysemy_examples

Contextual embeddings address this limitation by generating a different vector
for each token occurrence. They will be introduced later in the course.

# 18. Intrinsic Evaluation

**Intrinsic evaluation** measures the embedding space directly.

Examples:

- word-similarity correlation;
- word-relatedness correlation;
- analogy accuracy;
- nearest-neighbor inspection;
- clustering purity.

In [ ]:
human_similarity = pd.DataFrame(
    [
        ("doctor", "nurse", 0.90),
        ("teacher", "professor", 0.88),
        ("doctor", "teacher", 0.30),
        ("hospital", "clinic", 0.85),
        ("school", "university", 0.75),
    ],
    columns=["word_1", "word_2", "human_score"],
)

human_similarity["model_score"] = [
    cosine_between_words(
        row.word_1,
        row.word_2,
        matrix=dense_embeddings,
    )
    for row in human_similarity.itertuples(index=False)
]

human_similarity

In [ ]:
from scipy.stats import spearmanr

correlation, p_value = spearmanr(
    human_similarity["human_score"],
    human_similarity["model_score"],
)

print(f"Spearman correlation: {correlation:.3f}")
print(f"p-value: {p_value:.3f}")

The example contains too few pairs for a reliable conclusion. Formal intrinsic
evaluation requires established datasets and confidence analysis.

# 19. Extrinsic Evaluation

**Extrinsic evaluation** measures whether embeddings improve a downstream task.

Examples:

- text classification;
- NER;
- information retrieval;
- machine translation;
- semantic similarity;
- question answering.

An embedding may perform well on intrinsic similarity yet provide limited value
for a specific downstream task. Both evaluation levels are useful.

# 20. Bias and Ethical Considerations

Embeddings learn statistical associations from corpora. They may encode:

- gender stereotypes;
- racial or ethnic bias;
- occupational stereotypes;
- geographic bias;
- religious bias;
- domain-specific historical inequities.

In [ ]:
bias_audit_questions = pd.DataFrame(
    [
        ("Representation", "Which groups and language varieties appear?"),
        ("Association", "Do protected groups receive systematically different neighbors?"),
        ("Coverage", "Which names or dialects are out of vocabulary?"),
        ("Downstream impact", "Does bias affect ranking or classification?"),
        ("Documentation", "Are corpus and limitations clearly reported?"),
    ],
    columns=["Audit area", "Question"],
)

bias_audit_questions

Bias cannot be assessed from a few famous analogy examples alone. It requires
systematic, domain-aware evaluation.

# 21. Domain and Corpus Effects

Embeddings reflect the data used to construct them.

The word `cell` may be associated with:

- biology in biomedical text;
- phones in consumer technology;
- prisons in legal text;
- spreadsheets in office software.

In [ ]:
domain_examples = pd.DataFrame(
    [
        ("cell", "biomedical", "tissue, membrane, nucleus"),
        ("cell", "mobile technology", "phone, network, battery"),
        ("cell", "legal", "prison, inmate, detention"),
        ("cell", "spreadsheets", "row, column, formula"),
    ],
    columns=["Word", "Domain", "Likely neighbors"],
)

domain_examples

Pretrained embeddings should be evaluated on the target domain rather than
assumed to transfer perfectly.

# 22. Arabic and Multilingual Considerations

Distributional representations for Arabic must account for:

- rich morphology;
- clitics;
- diacritics;
- orthographic normalization;
- Modern Standard Arabic and dialects;
- code-switching;
- Arabizi;
- limited corpus coverage for some varieties.

Example surface forms:

```text
كتاب
الكتاب
والكتاب
كتابه
بالكتاب
```

Word-level embeddings may treat each form as a separate item.

In [ ]:
arabic_forms = pd.DataFrame(
    [
        ("كتاب", "book"),
        ("الكتاب", "the book"),
        ("والكتاب", "and the book"),
        ("كتابه", "his book"),
        ("بالكتاب", "with/by the book"),
    ],
    columns=["Arabic form", "Illustrative meaning"],
)

arabic_forms

Possible strategies include:

- light normalization;
- morphological segmentation;
- subword embeddings;
- character n-grams;
- separate models by variety;
- multilingual alignment.

## 22.1 Small Arabic Co-Occurrence Example

In [ ]:
arabic_corpus = [
    "الطبيب عالج المريض",
    "الممرضة فحصت المريض",
    "الطبيب يعمل في المستشفى",
    "الممرضة تعمل في العيادة",
    "المعلم يشرح الدرس",
    "الأستاذ يشرح الدرس",
    "المعلم يعمل في المدرسة",
    "الأستاذ يعمل في الجامعة",
]

arabic_tokens = [
    sentence.split()
    for sentence in arabic_corpus
]

arabic_vocabulary = sorted({
    token
    for sentence in arabic_tokens
    for token in sentence
})

arabic_index = {
    word: index
    for index, word in enumerate(arabic_vocabulary)
}

arabic_counts = np.zeros(
    (len(arabic_vocabulary), len(arabic_vocabulary)),
    dtype=float,
)

for sentence in arabic_tokens:
    for target, context in extract_contexts(sentence, window_size=2):
        arabic_counts[
            arabic_index[target],
            arabic_index[context],
        ] += 1

print("Arabic vocabulary size:", len(arabic_vocabulary))

In [ ]:
def arabic_similarity(
    word_a: str,
    word_b: str,
) -> float:
    vector_a = arabic_counts[
        arabic_index[word_a]
    ].reshape(1, -1)

    vector_b = arabic_counts[
        arabic_index[word_b]
    ].reshape(1, -1)

    return float(
        cosine_similarity(vector_a, vector_b)[0, 0]
    )


print(
    "الطبيب / الممرضة:",
    round(arabic_similarity("الطبيب", "الممرضة"), 3),
)

print(
    "المعلم / الأستاذ:",
    round(arabic_similarity("المعلم", "الأستاذ"), 3),
)

The corpus is too small for reliable Arabic embeddings; it only demonstrates
the distributional procedure.

# 23. Reproducibility and Reporting

Report:

- corpus source;
- corpus size;
- language and domain;
- tokenization;
- normalization;
- context definition;
- window size;
- weighting method;
- dimensionality;
- random seed;
- vocabulary threshold;
- evaluation datasets;
- known limitations.

In [ ]:
import platform
import sklearn

experiment_metadata = pd.Series(
    {
        "documents": len(corpus),
        "vocabulary_size": len(corpus_vocabulary),
        "context_window": window_size,
        "weighting": "PPMI",
        "reduction": "TruncatedSVD",
        "embedding_dimension": embedding_dimension,
        "random_state": 42,
        "python_version": platform.python_version(),
        "scikit_learn_version": sklearn.__version__,
    },
    name="Embedding experiment",
)

experiment_metadata

Reproducible embeddings require the corpus and vocabulary to be versioned, not
only the final matrix.

# 24. Knowledge Check

1. What is the distributional hypothesis?
2. Why do one-hot vectors provide no graded similarity?
3. What is a context window?
4. What information does a co-occurrence matrix contain?
5. How does cosine similarity differ from raw frequency?
6. Why can frequent words dominate count vectors?
7. What does PMI compare?
8. Why does PPMI replace negative values with zero?
9. Why is dimensionality reduction useful?
10. Why should individual dense dimensions not be over-interpreted?
11. What are nearest neighbors in an embedding space?
12. Why are word analogies approximate rather than guaranteed?
13. How does polysemy affect static embeddings?
14. How do intrinsic and extrinsic evaluations differ?
15. Which Arabic characteristics complicate word embeddings?

# 25. Exercises

## Exercise 1 — One-Hot Vectors

Construct one-hot vectors for a vocabulary of ten words and calculate pairwise
cosine similarity.

## Exercise 2 — Co-Occurrence Matrix

Build a co-occurrence matrix using window sizes 1, 2, and 5.

## Exercise 3 — Weighting

Compare raw counts, normalized counts, PMI, and PPMI.

## Exercise 4 — Nearest Neighbors

Inspect nearest neighbors before and after PPMI weighting.

## Exercise 5 — Dimensionality Reduction

Compare SVD dimensions 2, 5, 10, and 20.

## Exercise 6 — Intrinsic Evaluation

Create a small human similarity dataset and calculate Spearman correlation.

## Exercise 7 — Domain Comparison

Build separate embeddings from two domains and compare neighbors for ambiguous
words.

## Exercise 8 — Arabic Embeddings

Compare Arabic word forms before and after light normalization.

## Challenge Exercises

1. Implement distance-weighted context windows.
2. Build dependency-based contexts.
3. Align two embedding spaces with an orthogonal transformation.
4. Evaluate bias across names, occupations, and demographic terms.
5. Package the co-occurrence and SVD workflow as a reusable class.

# 26. Summary and Next Lesson

In this lesson:

- the distributional hypothesis connected meaning with context;
- one-hot vectors were distinguished from distributional vectors;
- context windows defined target-context observations;
- co-occurrence matrices represented words by neighboring words;
- cosine similarity compared vector directions;
- nearest neighbors exposed related words and corpus artifacts;
- PPMI emphasized informative associations;
- Truncated SVD converted sparse vectors into dense embeddings;
- two-dimensional projections supported exploration but distorted geometry;
- analogy arithmetic was treated as approximate;
- static embeddings were shown to mix word senses;
- intrinsic and extrinsic evaluation served different purposes;
- corpus bias and social bias required systematic auditing;
- Arabic embeddings required morphology-, script-, and variety-aware design.

## Next Lesson

**Lesson 22: Word2Vec — Skip-Gram and Continuous Bag-of-Words** introduces
predictive word embeddings, negative sampling, training pairs, similarity, and
practical model evaluation.

# References

- Harris, Z. S. *Distributional Structure*.
- Firth, J. R. *A Synopsis of Linguistic Theory*.
- Turney, P. D., & Pantel, P. *From Frequency to Meaning: Vector Space Models of Semantics*.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.
- distributional semantics, PMI, SVD, and word-similarity literature.